# 05 — Generate Migration Scenario Data

Builds the **migration QA scenario** used by notebook 06 (reconciliation).

- `raw.cr_businesses_source` — the "on-prem source system" (clean-ish, authoritative)
- `raw.cr_businesses_target` — the "migrated-into-Databricks" copy, with
  **deliberately injected migration drift** across 6 defect classes:
  1. Row loss (dropped during migration)
  2. Duplicate rows (re-run / double-load)
  3. Taxonomy change (industry values renamed)
  4. Schema drift (columns added / dropped)
  5. Value corruption (encoding, casing, truncation)
  6. Numeric drift (founded year altered)

Scenario framing: *IGT evaluating Costa Rica business partners for new
market entry; the partner registry is being migrated from an on-prem
system to Databricks and QA must validate the migration.*

Deterministic (fixed seed) so the repo is reproducible by anyone.

In [0]:
# ── Resolve runtime config written by notebook 01 ──────────────
def _load_pipeline_config():
    candidates = [
        "fieldops_dq.audit.pipeline_config",
        "workspace.fieldops_audit.pipeline_config",
    ]
    for tbl in candidates:
        try:
            rows = spark.table(tbl).collect()
            cfg = {row["config_key"]: row["config_value"] for row in rows}
            print(f"Loaded config from {tbl}")
            return cfg
        except Exception:
            continue
    raise RuntimeError(
        "pipeline_config not found. Run notebook 01 first — it resolves "
        "and persists the catalog/schema configuration."
    )

_cfg     = _load_pipeline_config()
CATALOG  = _cfg["CATALOG"]
RAW      = _cfg["RAW"]
print(f"  CATALOG={CATALOG} RAW={RAW}")

def r(table):
    return f"{CATALOG}.{RAW}.{table}"

Loaded config from workspace.fieldops_audit.pipeline_config
  CATALOG=workspace RAW=fieldops_raw


In [0]:
import random
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

# Scale: ~100K source rows. Adjust N_ROWS to taste.
N_ROWS = 100_000
SEED   = 20260519          # fixed seed → deterministic, reproducible

random.seed(SEED)

## 1 — Generate the SOURCE dataset
Same 10-column shape as the real Costa Rica businesses dataset:
`country, founded, id, industry, linkedin_url, locality, name,
region, size, website`.

In [0]:
# Reference pools for realistic-looking synthetic CR business data
INDUSTRIES = [
    "Information Technology", "Manufacturing", "Retail", "Financial Services",
    "Healthcare", "Tourism & Hospitality", "Agriculture", "Logistics",
    "Construction", "Professional Services", "Education", "Telecommunications",
]
CR_REGIONS = [
    "San José", "Alajuela", "Cartago", "Heredia",
    "Guanacaste", "Puntarenas", "Limón",
]
LOCALITIES = [
    "San José", "Escazú", "Heredia", "Alajuela", "Liberia", "Cartago",
    "Santa Ana", "Curridabat", "Tibás", "Desamparados", "Pérez Zeledón",
]
SIZE_BANDS = ["1-10", "11-50", "51-200", "201-500", "501-1000", "1001-5000"]

# Build rows in driver memory in chunks, then parallelize.
# (100K x 10 small string cols is fine for driver-side generation.)
def _make_source_rows(n):
    rows = []
    for i in range(1, n + 1):
        name = f"CR Business {i:06d} S.A."
        rows.append((
            "Costa Rica",                                  # country
            random.randint(1950, 2024),                    # founded (int)
            f"CRB-{i:06d}",                                 # id (business key)
            random.choice(INDUSTRIES),                      # industry
            f"https://linkedin.com/company/crb-{i:06d}",    # linkedin_url
            random.choice(LOCALITIES),                      # locality
            name,                                           # name
            random.choice(CR_REGIONS),                      # region
            random.choice(SIZE_BANDS),                      # size
            f"https://crb{i:06d}.co.cr",                    # website
        ))
    return rows

source_rows = _make_source_rows(N_ROWS)

source_schema = StructType([
    StructField("country",      StringType(),  True),
    StructField("founded",      IntegerType(), True),
    StructField("id",           StringType(),  True),
    StructField("industry",     StringType(),  True),
    StructField("linkedin_url", StringType(),  True),
    StructField("locality",     StringType(),  True),
    StructField("name",         StringType(),  True),
    StructField("region",       StringType(),  True),
    StructField("size",         StringType(),  True),
    StructField("website",      StringType(),  True),
])

df_source = spark.createDataFrame(source_rows, schema=source_schema)
df_source.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(r("cr_businesses_source"))

src_count = spark.table(r("cr_businesses_source")).count()
print(f"SOURCE written: {r('cr_businesses_source')} = {src_count:,} rows")

SOURCE written: workspace.fieldops_raw.cr_businesses_source = 100,000 rows


## 2 — Generate the TARGET dataset (with injected migration drift)

We start from an exact copy of source, then apply 6 realistic
migration defects. Each is logged here so notebook 06's reconciliation
can be checked against the *known* injected truth (a test oracle).

In [0]:
from pyspark.sql import functions as F

tgt = spark.table(r("cr_businesses_source"))

# ---- DEFECT 1: ROW LOSS — drop ~0.5% of rows (data lost in migration) ----
# Deterministic: drop rows whose numeric id is divisible by 199.
tgt = tgt.withColumn("_idnum", F.regexp_replace("id", "CRB-", "").cast("int"))
n_before = tgt.count()
tgt_lost = tgt.filter(~(F.col("_idnum") % 199 == 0))
dropped = n_before - tgt_lost.count()
print(f"DEFECT 1 (row loss): dropped {dropped:,} rows (id % 199 == 0)")

# ---- DEFECT 2: TAXONOMY CHANGE — rename two industry values ----
# 'Information Technology' -> 'IT', 'Tourism & Hospitality' -> 'Tourism'
# Classic migration issue: source taxonomy not preserved in target.
tgt_tax = tgt_lost.withColumn(
    "industry",
    F.when(F.col("industry") == "Information Technology", F.lit("IT"))
     .when(F.col("industry") == "Tourism & Hospitality",  F.lit("Tourism"))
     .otherwise(F.col("industry"))
)
print("DEFECT 2 (taxonomy): 'Information Technology'->'IT', "
      "'Tourism & Hospitality'->'Tourism'")

# ---- DEFECT 3: VALUE CORRUPTION — casing + truncation on a slice ----
# ~0.3% of rows: name upper-cased AND website truncated (lost characters).
tgt_corrupt = tgt_tax.withColumn(
    "name",
    F.when(F.col("_idnum") % 311 == 0, F.upper(F.col("name")))
     .otherwise(F.col("name"))
).withColumn(
    "website",
    F.when(F.col("_idnum") % 311 == 0,
           F.regexp_replace(F.col("website"), r"\.co\.cr$", ".co"))
     .otherwise(F.col("website"))
)
print("DEFECT 3 (corruption): name upper-cased + website truncated "
      "where id % 311 == 0")

# ---- DEFECT 4: NUMERIC DRIFT — shift 'founded' by +1 on a slice ----
# ~0.2% of rows: founded year off by one (silent numeric corruption that
# row counts and hashes-of-keys would miss — caught by aggregate checks).
tgt_numeric = tgt_corrupt.withColumn(
    "founded",
    F.when(F.col("_idnum") % 467 == 0, F.col("founded") + F.lit(1))
     .otherwise(F.col("founded"))
)
print("DEFECT 4 (numeric drift): founded += 1 where id % 467 == 0")

# ---- DEFECT 5: SCHEMA DRIFT — drop a column, add two new ones ----
# Target dropped 'linkedin_url' and added 'migrated_at' + 'source_system'.
# Very common: target schema not identical to source.
tgt_schema = (tgt_numeric
    .drop("linkedin_url")
    .drop("_idnum")
    .withColumn("migrated_at",   F.current_timestamp())
    .withColumn("source_system", F.lit("ONPREM_LEGACY"))
)
print("DEFECT 5 (schema drift): dropped 'linkedin_url'; "
      "added 'migrated_at','source_system'")

# ---- DEFECT 6: DUPLICATES — re-append ~0.1% of rows ----
# Simulates a partial re-run during migration (non-idempotent load).
dupe_sample = tgt_schema.filter(
    F.regexp_replace("id", "CRB-", "").cast("int") % 991 == 0
)
tgt_final = tgt_schema.unionByName(dupe_sample)
n_dupes = dupe_sample.count()
print(f"DEFECT 6 (duplicates): re-appended {n_dupes:,} rows (id % 991 == 0)")

# ---- Persist target ----
tgt_final.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(r("cr_businesses_target"))

tgt_count = spark.table(r("cr_businesses_target")).count()
print(f"\nTARGET written: {r('cr_businesses_target')} = {tgt_count:,} rows")

DEFECT 1 (row loss): dropped 502 rows (id % 199 == 0)
DEFECT 2 (taxonomy): 'Information Technology'->'IT', 'Tourism & Hospitality'->'Tourism'
DEFECT 3 (corruption): name upper-cased + website truncated where id % 311 == 0
DEFECT 4 (numeric drift): founded += 1 where id % 467 == 0
DEFECT 5 (schema drift): dropped 'linkedin_url'; added 'migrated_at','source_system'
DEFECT 6 (duplicates): re-appended 100 rows (id % 991 == 0)

TARGET written: workspace.fieldops_raw.cr_businesses_target = 99,598 rows


## 3 — Injected-defect manifest (the test oracle)

notebook 06 should *independently rediscover* these numbers. We persist
the expected truth so the reconciliation can be scored for accuracy
("did QA catch what we know we broke?").

In [0]:
from datetime import datetime, timezone

manifest = spark.createDataFrame([
    ("row_loss",      "rows dropped (id % 199 == 0)",            int(dropped)),
    ("taxonomy",      "industry values renamed",                 2),
    ("corruption",    "name/website corrupted (id % 311 == 0)",  -1),
    ("numeric_drift", "founded += 1 (id % 467 == 0)",            -1),
    ("schema_drift",  "linkedin_url dropped; 2 cols added",       3),
    ("duplicates",    "rows duplicated (id % 991 == 0)",         int(n_dupes)),
], "defect_class STRING, description STRING, expected_count INT")

manifest = manifest.withColumn("generated_at", F.lit(datetime.now(timezone.utc)))
manifest.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(r("cr_migration_manifest"))

print("Injected-defect manifest:")
for row in manifest.collect():
    exp = row["expected_count"]
    exp_s = "(measured in 06)" if exp == -1 else f"{exp:,}"
    print(f"  {row['defect_class']:<14} {exp_s:>18}  — {row['description']}")

print(f"\nSOURCE = {src_count:,} rows | TARGET = {tgt_count:,} rows | "
      f"net delta = {tgt_count - src_count:,}")
print("Next: run 06_migration_reconciliation")

Injected-defect manifest:
  row_loss                      502  — rows dropped (id % 199 == 0)
  taxonomy                        2  — industry values renamed
  corruption       (measured in 06)  — name/website corrupted (id % 311 == 0)
  numeric_drift    (measured in 06)  — founded += 1 (id % 467 == 0)
  schema_drift                    3  — linkedin_url dropped; 2 cols added
  duplicates                    100  — rows duplicated (id % 991 == 0)

SOURCE = 100,000 rows | TARGET = 99,598 rows | net delta = -402
Next: run 06_migration_reconciliation
